In [ ]:
!pip install -q pandas numpy scikit-learn tensorflow nltk datasets transformers accelerate torch


In [ ]:
import os
import re
import random
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("TensorFlow version:", tf.__version__)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


TensorFlow version: 2.20.0
PyTorch version: 2.10.0+cu128
CUDA available: True


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving CEAS_08_cleaned.csv to CEAS_08_cleaned.csv
Saving machinewars_filtered_emails.json to machinewars_filtered_emails.json
Saving Nazario_cleaned.csv to Nazario_cleaned.csv
Saving Nigerian_Fraud_cleaned.csv to Nigerian_Fraud_cleaned.csv
Saving SpamAssasin_cleaned.csv to SpamAssasin_cleaned.csv


In [ ]:
import pandas as pd
import json
import re
from pathlib import Path


# ---------------------------------
# 1. Helpers
# ---------------------------------
def safe_str(x):
    if pd.isna(x):
        return ""
    s = str(x).strip()
    if s.lower() in {"null", "none", "nan"}:
        return ""
    return s


URL_PATTERN = re.compile(
    r"((?:https?://|www\.)[^\s<>\"'()]+)",
    re.IGNORECASE,
)


def extract_urls_from_text(text):
    text = safe_str(text)
    matches = URL_PATTERN.findall(text)

    seen = set()
    urls = []
    for url in matches:
        url = url.strip().rstrip('.,;:!?')
        if url and url not in seen:
            seen.add(url)
            urls.append(url)

    return urls


def normalize_url_value(value):
    if isinstance(value, list):
        value = " | ".join(safe_str(v) for v in value if safe_str(v))
    return safe_str(value)


def populate_url_column(df):
    if "url" not in df.columns:
        df["url"] = ""

    df["url"] = df["url"].apply(normalize_url_value)

    missing_mask = df["url"].eq("")
    if missing_mask.any():
        df.loc[missing_mask, "url"] = df.loc[missing_mask, "body"].apply(
            lambda body: " | ".join(extract_urls_from_text(body))
        )

    return df


def normalize_sender(sender):
    return safe_str(sender).lower()


def build_text_all_fields(row):
    sender = normalize_sender(row.get("sender", ""))
    subject = safe_str(row.get("subject", ""))
    body = safe_str(row.get("body", ""))
    url = safe_str(row.get("url", ""))

    return (
        f"[SENDER] {sender}\n"
        f"[SUBJECT] {subject}\n"
        f"[BODY] {body}\n"
        f"[URL] {url}"
    ).strip()


def build_text_all_fields_from_parts(sender, subject, body, url):
    return build_text_all_fields({
        "sender": sender,
        "subject": subject,
        "body": body,
        "url": url,
    })


# ---------------------------------
# 2. Label handling
# ---------------------------------
def normalize_machinewars_label(label, spam_as_phishing=False):
    """
    MachineWars:
      Phishing -> 1
      Legitimate/Valid/Ham -> 0
      Spam -> 1 if spam_as_phishing=True else excluded
    """
    if pd.isna(label):
        return None

    label = str(label).strip().lower()

    if label == "phishing":
        return 1

    if label in {"legitimate", "valid", "ham", "benign", "safe"}:
        return 0

    if label == "spam":
        return 1 if spam_as_phishing else None

    return None


def normalize_test_label(label):
    """
    For CEAS-style test sets.
    Spam is excluded here unless you explicitly want otherwise.
    """
    if pd.isna(label):
        return None

    label = str(label).strip().lower()

    if label == "phishing":
        return 1

    if label in {"legitimate", "valid", "ham", "benign", "safe"}:
        return 0

    return None


# ---------------------------------
# 3. MachineWars loader
# ---------------------------------
def load_machinewars(json_path_or_list, spam_as_phishing=False, dataset_name="machinewars"):
    """
    Expected MachineWars fields:
      sender, subject, body, type, url
    """
    if isinstance(json_path_or_list, (str, Path)):
        with open(json_path_or_list, "r", encoding="utf-8") as f:
            data = json.load(f)
    else:
        data = json_path_or_list

    df = pd.DataFrame(data).copy()

    if "type" in df.columns:
        df = df.rename(columns={"type": "label"})

    for col in ["subject", "body", "label"]:
        if col not in df.columns:
            df[col] = ""

    if "sender" not in df.columns:
        df["sender"] = ""

    df = populate_url_column(df)

    df["dataset"] = dataset_name
    df["label_raw"] = df["label"].astype(str).str.strip().str.lower()
    df["label_id"] = df["label_raw"].apply(
        lambda x: normalize_machinewars_label(x, spam_as_phishing=spam_as_phishing)
    )

    df = df[df["label_id"].notna()].copy()
    df["label_id"] = df["label_id"].astype(int)
    df["label"] = df["label_id"].map({0: "legitimate", 1: "phishing"})
    df["text"] = df.apply(build_text_all_fields, axis=1)

    df = df[[
        "dataset", "sender", "subject", "body", "url",
        "label_raw", "label", "label_id", "text"
    ]]

    return df


# ---------------------------------
# 4. CEAS-style test loader
# ---------------------------------
def load_ceas_style_csv(csv_path, dataset_name=None):
    """
    Assumes CEAS-style columns similar to:
      subject, body, label
    """
    csv_path = Path(csv_path)
    if dataset_name is None:
        dataset_name = csv_path.stem

    df = pd.read_csv(csv_path).copy()

    rename_map = {
        "Subject": "subject",
        "Body": "body",
        "Label": "label",
        "type": "label",
        "From": "sender",
        "Sender": "sender",
    }
    df = df.rename(columns=rename_map)

    for col in ["subject", "body", "label"]:
        if col not in df.columns:
            df[col] = ""

    if "sender" not in df.columns:
        df["sender"] = ""

    df = populate_url_column(df)

    df["dataset"] = dataset_name
    df["label_raw"] = df["label"].astype(str).str.strip().str.lower()
    df["label_id"] = df["label_raw"].apply(normalize_test_label)

    df = df[df["label_id"].notna()].copy()
    df["label_id"] = df["label_id"].astype(int)
    df["label"] = df["label_id"].map({0: "legitimate", 1: "phishing"})
    df["text"] = df.apply(build_text_all_fields, axis=1)

    df = df[[
        "dataset", "sender", "subject", "body", "url",
        "label_raw", "label", "label_id", "text"
    ]]

    return df


In [ ]:
machinewars_spam_as_phishing_df = load_machinewars(
    "machinewars_filtered_emails.json",
    spam_as_phishing=True,
    dataset_name="machinewars"
)
train_df, val_df = train_test_split(
    machinewars_spam_as_phishing_df,
    test_size=0.2,
    random_state=SEED,
    stratify=machinewars_spam_as_phishing_df["label_id"]
)

In [ ]:
from datasets import Dataset

train_ds = Dataset.from_pandas(
    train_df[["text", "label_id"]]
    .rename(columns={"label_id": "labels"})
    .reset_index(drop=True)
)

val_ds = Dataset.from_pandas(
    val_df[["text", "label_id"]]
    .rename(columns={"label_id": "labels"})
    .reset_index(drop=True)
)
test_paths = [
    "CEAS_08_cleaned.csv",
    "Nazario_cleaned.csv",
    "Nigerian_Fraud_cleaned.csv",
    "SpamAssasin_cleaned.csv",
]

test_dfs = [load_ceas_style_csv(p) for p in test_paths]

for i, df in enumerate(test_dfs, 1):
    print(f"\nTest dataset {i}:")
    print(df["label"].value_counts())
    print(df.head(2))


Test dataset 1:
label
phishing      21842
legitimate    17312
Name: count, dtype: int64
           dataset                            sender  \
0  CEAS_08_cleaned  Young Esposito <Young@iworld.de>   
1  CEAS_08_cleaned      Mok <ipline's1983@icable.ph>   

                     subject  \
0  Never agree to be a loser   
1     Befriend Jenna Jameson   

                                                body  \
0  Buck up, your troubles caused by small dimensi...   
1  \nUpgrade your sex and pleasures with these te...   

                         url label_raw     label  label_id  \
0      http://whitedone.com/  phishing  phishing         1   
1  http://www.brightmade.com  phishing  phishing         1   

                                                text  
0  [SENDER] young esposito <young@iworld.de>\n[SU...  
1  [SENDER] mok <ipline's1983@icable.ph>\n[SUBJEC...  

Test dataset 2:
label
phishing    1565
Name: count, dtype: int64
           dataset                                        

In [ ]:
def compute_binary_metrics(y_true, y_pred, y_prob):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="binary",
        zero_division=0
    )
    acc = accuracy_score(y_true, y_pred)

    try:
        roc_auc = roc_auc_score(y_true, y_prob)
    except Exception:
        roc_auc = float("nan")

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
    }
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    # If model returns tuple-like predictions, keep logits only.
    if isinstance(logits, tuple):
        logits = logits[0]

    y_true = labels
    y_pred = np.argmax(logits, axis=-1)

    # Convert logits to probability for class 1.
    exp_logits = np.exp(logits - np.max(logits, axis=1, keepdims=True))
    probs = exp_logits / exp_logits.sum(axis=1, keepdims=True)
    y_prob = probs[:, 1]

    return compute_binary_metrics(y_true, y_pred, y_prob)

In [ ]:
X_train = train_df["text"].astype(str).tolist()
y_train = train_df["label_id"].astype(int).values

X_val = val_df["text"].astype(str).tolist()
y_val = val_df["label_id"].astype(int).values


DISTILBERT_MODEL_NAME = "distilbert-base-uncased"
DISTILBERT_MAX_LENGTH = 512

distilbert_tokenizer = AutoTokenizer.from_pretrained(DISTILBERT_MODEL_NAME)

distilbert_model = AutoModelForSequenceClassification.from_pretrained(
    DISTILBERT_MODEL_NAME,
    num_labels=2,
)


train_texts = [str(x) for x in X_train]
eval_texts = [str(x) for x in X_val]

train_labels = [int(y) for y in y_train]
eval_labels = [int(y) for y in y_val]

train_dataset = Dataset.from_dict({
    "text": train_texts,
    "labels": train_labels,
})

eval_dataset = Dataset.from_dict({
    "text": eval_texts,
    "labels": eval_labels,
})

def tokenize_distilbert_batch(batch):
    return distilbert_tokenizer(
        batch["text"],
        truncation=True,
        padding=False,
        max_length=DISTILBERT_MAX_LENGTH,
    )

train_dataset = train_dataset.map(
    tokenize_distilbert_batch,
    batched=True,
    remove_columns=["text"],
)

eval_dataset = eval_dataset.map(
    tokenize_distilbert_batch,
    batched=True,
    remove_columns=["text"],
)

train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"],
)

eval_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"],
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/15840 [00:00<?, ? examples/s]

Map:   0%|          | 0/3960 [00:00<?, ? examples/s]

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
DISTILBERT_BATCH_SIZE = 32
DISTILBERT_EPOCHS = 3

# DistilBERT has no token_type_ids; truncation is important because emails can be long.


def tokenize_distilbert_batch(batch):
    return distilbert_tokenizer(
        batch["text"],
        truncation=True,
        max_length=DISTILBERT_MAX_LENGTH,
    )


train_ds_tok = train_ds.map(tokenize_distilbert_batch, batched=True)
val_ds_tok = val_ds.map(tokenize_distilbert_batch, batched=True)

# Keep only tensors needed by Trainer.
train_ds_tok = train_ds_tok.remove_columns([c for c in train_ds_tok.column_names if c not in {"input_ids", "attention_mask", "labels"}])
val_ds_tok = val_ds_tok.remove_columns([c for c in val_ds_tok.column_names if c not in {"input_ids", "attention_mask", "labels"}])

id2label = {0: "legitimate", 1: "phishing"}
label2id = {v: k for k, v in id2label.items()}

distilbert_model = AutoModelForSequenceClassification.from_pretrained(
    DISTILBERT_MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

data_collator = DataCollatorWithPadding(tokenizer=distilbert_tokenizer)

# Keep the SVM's class_weight="balanced" behavior by weighting cross-entropy.
class_counts = np.bincount(y_train, minlength=2)
class_weights = len(y_train) / (2.0 * np.maximum(class_counts, 1))
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)

classes = np.array([0, 1])

class_weights_np = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=np.array(train_labels),
)

class_weights = torch.tensor(class_weights_np, dtype=torch.float)
print("Class weights:", class_weights)


class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        weights = self.class_weights.to(outputs.logits.device) if self.class_weights is not None else None
        loss_fct = torch.nn.CrossEntropyLoss(weight=weights)
        loss = loss_fct(outputs.logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


def compute_distilbert_trainer_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
    preds = (probs >= 0.5).astype(int)
    return compute_binary_metrics(labels, preds, probs)



# Transformers renamed evaluation_strategy to eval_strategy in newer releases.
# This small compatibility shim lets the notebook run on both old and new Colab images.
import inspect
training_args_kwargs = dict(
    output_dir="/content/distilbert_phishing_model",
    learning_rate=2e-5,
    per_device_train_batch_size=DISTILBERT_BATCH_SIZE,
    per_device_eval_batch_size=DISTILBERT_BATCH_SIZE,
    num_train_epochs=DISTILBERT_EPOCHS,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="no",
    report_to="none",
    seed=SEED,
)
if "eval_strategy" in inspect.signature(TrainingArguments.__init__).parameters:
    training_args_kwargs["eval_strategy"] = "epoch"
else:
    training_args_kwargs["evaluation_strategy"] = "epoch"

training_args = TrainingArguments(**training_args_kwargs)

distilbert_trainer = WeightedTrainer(
    model=distilbert_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
)

distilbert_trainer.train()

print("DistilBERT training complete.")


Map:   0%|          | 0/15840 [00:00<?, ? examples/s]

Map:   0%|          | 0/3960 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Class weights: tensor([1.5000, 0.7500])


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Roc Auc
1,0.108524,0.097182,0.966919,0.980100,0.970076,0.975062,0.994420
2,0.093606,0.081582,0.980556,0.983038,0.987879,0.985452,0.997299
3,0.027098,0.092310,0.980051,0.978692,0.991667,0.985136,0.998276


DistilBERT training complete.


In [ ]:
def distilbert_predict_one(text):
    pred, prob = distilbert_batch_predict([text], batch_size=1)
    return int(pred[0]), float(prob[0])


def distilbert_batch_predict(texts, batch_size=32):
    texts = [str(t) for t in texts]
    if len(texts) == 0:
        return np.array([], dtype=int), np.array([], dtype=float)

    distilbert_model.eval()
    all_probs = []
    device = distilbert_model.device

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]
        encoded = distilbert_tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=DISTILBERT_MAX_LENGTH,
            return_tensors="pt",
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}

        with torch.no_grad():
            logits = distilbert_model(**encoded).logits
            probs = torch.softmax(logits, dim=-1)[:, 1]

        all_probs.append(probs.detach().cpu().numpy())

    probs = np.concatenate(all_probs)
    preds = (probs >= 0.5).astype(int)
    return preds, probs


def evaluate_distilbert(df_eval):
    y_true = df_eval["label_id"].astype(int).values
    y_pred, y_prob = distilbert_batch_predict(df_eval["text"].astype(str).tolist())
    return compute_binary_metrics(y_true, y_pred, y_prob)



In [ ]:
print("Validation metrics:")
print(evaluate_distilbert(val_df))

distilbert_rows = [{"dataset": "validation", **evaluate_distilbert(val_df)}]

for test_df in test_dfs:
    test_name = test_df["dataset"].iloc[0]
    metrics = evaluate_distilbert(test_df)
    distilbert_rows.append({"dataset": test_name, **metrics})

distilbert_results_df = pd.DataFrame(distilbert_rows)
distilbert_results_df


Validation metrics:
{'accuracy': 0.9800505050505051, 'precision': 0.9786915887850467, 'recall': 0.9916666666666667, 'f1': 0.9851364063969896, 'roc_auc': np.float64(0.9982763716712579)}


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


,dataset,accuracy,precision,recall,f1,roc_auc
0,validation,0.980051,0.978692,0.991667,0.985136,0.998276
1,CEAS_08_cleaned,0.770700,0.973289,0.605576,0.746613,0.958874
2,Nazario_cleaned,0.959744,1.000000,0.959744,0.979459,NaN
3,Nigerian_Fraud_cleaned,0.882953,1.000000,0.882953,0.937839,NaN
4,SpamAssasin_cleaned,0.863660,0.906854,0.600698,0.722689,0.940701


In [ ]:
import nltk
from nltk.corpus import wordnet

nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [ ]:
def benign_prefix_attack(text):
    prefix = (
        "Hello,\n\n"
        "This is a routine business communication regarding normal account activity. "
        "Please review the information below.\n\n"
    )
    return prefix + str(text)


def benign_suffix_attack(text):
    suffix = (
        "\n\nThank you for your attention."
        "\nThis message is part of a standard informational workflow."
        "\nBest regards."
    )
    return str(text) + suffix


def contradiction_attack(text):
    prefix = (
        "Important note: this is a legitimate and safe message. "
        "It does not request any unsafe action.\n\n"
    )
    return prefix + str(text)


def prefix_injection_attack(text):
    prefix = (
        "This is a normal and trustworthy business email. "
        "The message is legitimate, safe, and routine.\n\n"
    )
    return prefix + str(text)


PHISHING_KEYWORDS = {
    "verify", "verification", "account", "password", "login", "signin",
    "security", "alert", "urgent", "confirm", "suspend", "suspended",
    "click", "update", "reset", "limited", "immediately"
}

def keyword_deletion_attack(text, max_delete=5):
    words = str(text).split()
    new_words = []
    deleted = 0

    for w in words:
        clean = re.sub(r"^[^\w]+|[^\w]+$", "", w).lower()
        if clean in PHISHING_KEYWORDS and deleted < max_delete:
            deleted += 1
            continue
        new_words.append(w)

    return " ".join(new_words)


def get_synonyms(word):
    syns = set()
    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            s = lemma.name().replace("_", " ").strip()
            if s and s.lower() != word.lower():
                syns.add(s)
    return list(syns)


def synonym_attack(text, replace_prob=0.12, max_replacements=8, seed=42):
    rng = random.Random(seed)
    words = str(text).split()
    new_words = []
    replacements = 0

    for w in words:
        clean = re.sub(r"^[^\w]+|[^\w]+$", "", w)

        if (
            replacements < max_replacements
            and len(clean) >= 4
            and clean.isalpha()
            and rng.random() < replace_prob
        ):
            syns = get_synonyms(clean)
            syns = [s for s in syns if s.isalpha() and len(s.split()) == 1]

            if syns:
                replacement = rng.choice(syns)
                if w.istitle():
                    replacement = replacement.title()
                new_words.append(replacement)
                replacements += 1
                continue

        new_words.append(w)

    return " ".join(new_words)

In [ ]:
predict_one = distilbert_predict_one
batch_predict = distilbert_batch_predict
ACTIVE_MODEL_NAME = "distilbert"
print("Active model:", ACTIVE_MODEL_NAME)


Active model: distilbert


In [ ]:
def apply_attack_to_fields(row, subject_attack_fn=None, body_attack_fn=None):
    sender = row.get("sender", "")
    subject = safe_str(row.get("subject", ""))
    body = safe_str(row.get("body", ""))
    url = safe_str(row.get("url", ""))

    if subject_attack_fn is not None:
        subject = subject_attack_fn(subject)

    if body_attack_fn is not None:
        body = body_attack_fn(body)

    return build_text_all_fields_from_parts(sender, subject, body, url)


def evaluate_attack_common(df_eval, attack_name, row_attack_fn):
    attacked_texts = [row_attack_fn(row) for _, row in df_eval.iterrows()]
    y_true = df_eval["label_id"].astype(int).values

    y_pred, y_prob = batch_predict(attacked_texts)

    return {
        "attack": attack_name,
        "n_samples": len(df_eval),
        **compute_binary_metrics(y_true, y_pred, y_prob)
    }


In [ ]:
attack_rows_val = []

attack_rows_val.append(evaluate_attack_common(val_df, "clean", lambda row: row["text"]))
attack_rows_val.append(evaluate_attack_common(val_df, "benign_prefix", lambda row: apply_attack_to_fields(row, body_attack_fn=benign_prefix_attack)))
attack_rows_val.append(evaluate_attack_common(val_df, "benign_suffix", lambda row: apply_attack_to_fields(row, body_attack_fn=benign_suffix_attack)))
attack_rows_val.append(evaluate_attack_common(val_df, "contradiction", lambda row: apply_attack_to_fields(row, body_attack_fn=contradiction_attack)))
attack_rows_val.append(
    evaluate_attack_common(
        val_df,
        "synonym_attack",
        lambda row: apply_attack_to_fields(
            row,
            subject_attack_fn=lambda x: synonym_attack(x, replace_prob=0.12, max_replacements=8, seed=42),
            body_attack_fn=lambda x: synonym_attack(x, replace_prob=0.12, max_replacements=8, seed=42),
        )
    )
)
attack_rows_val.append(
    evaluate_attack_common(
        val_df,
        "keyword_deletion",
        lambda row: apply_attack_to_fields(
            row,
            subject_attack_fn=lambda x: keyword_deletion_attack(x, max_delete=5),
            body_attack_fn=lambda x: keyword_deletion_attack(x, max_delete=5),
        )
    )
)
attack_rows_val.append(evaluate_attack_common(val_df, "prefix_injection", lambda row: apply_attack_to_fields(row, body_attack_fn=prefix_injection_attack)))

attack_results_val_df = pd.DataFrame(attack_rows_val).sort_values("f1", ascending=False)
attack_results_val_df


,attack,n_samples,accuracy,precision,recall,f1,roc_auc
3,contradiction,3960,0.981313,0.980885,0.991288,0.986059,0.998223
5,keyword_deletion,3960,0.980303,0.980135,0.990530,0.985305,0.998029
1,benign_prefix,3960,0.980051,0.976554,0.993939,0.985170,0.998022
0,clean,3960,0.980051,0.978692,0.991667,0.985136,0.998276
2,benign_suffix,3960,0.979798,0.980480,0.989394,0.984917,0.998285
4,synonym_attack,3960,0.978535,0.977927,0.990152,0.984002,0.998020
6,prefix_injection,3960,0.977525,0.971534,0.995455,0.983349,0.997982


In [ ]:
all_attack_tables = {}

for test_df in test_dfs:
    test_name = test_df["dataset"].iloc[0]

    rows = []
    rows.append(evaluate_attack_common(test_df, "clean", lambda row: row["text"]))
    rows.append(evaluate_attack_common(test_df, "benign_prefix", lambda row: apply_attack_to_fields(row, body_attack_fn=benign_prefix_attack)))
    rows.append(evaluate_attack_common(test_df, "benign_suffix", lambda row: apply_attack_to_fields(row, body_attack_fn=benign_suffix_attack)))
    rows.append(evaluate_attack_common(test_df, "contradiction", lambda row: apply_attack_to_fields(row, body_attack_fn=contradiction_attack)))
    rows.append(
        evaluate_attack_common(
            test_df,
            "synonym_attack",
            lambda row: apply_attack_to_fields(
                row,
                subject_attack_fn=lambda x: synonym_attack(x, replace_prob=0.12, max_replacements=8, seed=42),
                body_attack_fn=lambda x: synonym_attack(x, replace_prob=0.12, max_replacements=8, seed=42),
            )
        )
    )
    rows.append(
        evaluate_attack_common(
            test_df,
            "keyword_deletion",
            lambda row: apply_attack_to_fields(
                row,
                subject_attack_fn=lambda x: keyword_deletion_attack(x, max_delete=5),
                body_attack_fn=lambda x: keyword_deletion_attack(x, max_delete=5),
            )
        )
    )
    rows.append(evaluate_attack_common(test_df, "prefix_injection", lambda row: apply_attack_to_fields(row, body_attack_fn=prefix_injection_attack)))

    result_df = pd.DataFrame(rows).sort_values("f1", ascending=False)
    all_attack_tables[test_name] = result_df

    print(f"\n=== {ACTIVE_MODEL_NAME} | {test_name} ===")
    print(result_df)



=== distilbert | CEAS_08_cleaned ===
             attack  n_samples  accuracy  precision    recall        f1  \
6  prefix_injection      39154  0.776191   0.968748  0.618762  0.755176   
4    synonym_attack      39154  0.770598   0.970924  0.606950  0.746957   
5  keyword_deletion      39154  0.770905   0.974072  0.605439  0.746739   
0             clean      39154  0.770700   0.973289  0.605576  0.746613   
1     benign_prefix      39154  0.745620   0.967207  0.563089  0.711789   
3     contradiction      39154  0.742197   0.974552  0.552285  0.705026   
2     benign_suffix      39154  0.722302   0.973496  0.516253  0.674705   

    roc_auc  
6  0.959759  
4  0.957098  
5  0.958804  
0  0.958874  
1  0.952536  
3  0.961001  
2  0.955915  


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist


=== distilbert | Nazario_cleaned ===
             attack  n_samples  accuracy  precision    recall        f1  \
6  prefix_injection       1565  0.962300        1.0  0.962300  0.980788   
1     benign_prefix       1565  0.961022        1.0  0.961022  0.980124   
0             clean       1565  0.959744        1.0  0.959744  0.979459   
4    synonym_attack       1565  0.953994        1.0  0.953994  0.976455   
3     contradiction       1565  0.953994        1.0  0.953994  0.976455   
5  keyword_deletion       1565  0.944409        1.0  0.944409  0.971410   
2     benign_suffix       1565  0.939936        1.0  0.939936  0.969038   

   roc_auc  
6      NaN  
1      NaN  
0      NaN  
4      NaN  
3      NaN  
5      NaN  
2      NaN  


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist


=== distilbert | Nigerian_Fraud_cleaned ===
             attack  n_samples  accuracy  precision    recall        f1  \
6  prefix_injection       3332  0.885354        1.0  0.885354  0.939191   
4    synonym_attack       3332  0.884454        1.0  0.884454  0.938685   
0             clean       3332  0.882953        1.0  0.882953  0.937839   
5  keyword_deletion       3332  0.882053        1.0  0.882053  0.937331   
2     benign_suffix       3332  0.877251        1.0  0.877251  0.934612   
3     contradiction       3332  0.867947        1.0  0.867947  0.929306   
1     benign_prefix       3332  0.846939        1.0  0.846939  0.917127   

   roc_auc  
6      NaN  
4      NaN  
0      NaN  
5      NaN  
2      NaN  
3      NaN  
1      NaN  

=== distilbert | SpamAssasin_cleaned ===
             attack  n_samples  accuracy  precision    recall        f1  \
6  prefix_injection       5809  0.874161   0.912971  0.635041  0.749056   
1     benign_prefix       5809  0.871751   0.922676  0.618

In [ ]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

In [ ]:
def benign_prefix_attack(text):
    prefix = (
        "Hello,\n\n"
        "This is a routine business communication regarding normal account activity. "
        "Please review the information below.\n\n"
    )
    return prefix + str(text)


def benign_suffix_attack(text):
    suffix = (
        "\n\nThank you for your attention."
        "\nThis message is part of a standard informational workflow."
        "\nBest regards."
    )
    return str(text) + suffix


def contradiction_attack(text):
    prefix = (
        "Important note: this is a legitimate and safe message. "
        "It does not request any unsafe action.\n\n"
    )
    return prefix + str(text)


def training_context_attack(text):
    prefix = (
        "Example of a safe enterprise email used for routine staff communication:\n\n"
    )
    return prefix + str(text)


def noise_injection_attack(text):
    noise = (
        "\n\nReference ID: A7K2-91LM"
        "\nDepartment: operations"
        "\nCase Type: routine review"
    )
    return str(text) + noise

In [ ]:
import re

STOPWORDS_ATTACK = {
    "the", "a", "an", "and", "or", "to", "of", "in", "for", "on", "at",
    "is", "are", "this", "that", "with", "from", "by", "as", "it", "be",
    "was", "were", "subject", "body", "sender", "url", "dear", "hello", "hi", "regards",
    "thanks", "thank", "best"
}

PROTECTED_TOKENS = {"[SENDER]", "[SUBJECT]", "[BODY]", "[URL]"}


def basic_tokenize_with_indices(text):
    """
    Splits text into whitespace-separated tokens and keeps positions.
    """
    tokens = str(text).split()
    return tokens


def is_deletable_token(tok):
    if tok in PROTECTED_TOKENS:
        return False

    clean = re.sub(r"^[^\w]+|[^\w]+$", "", tok).lower()
    if len(clean) < 3:
        return False
    if clean in STOPWORDS_ATTACK:
        return False
    if not any(ch.isalpha() for ch in clean):
        return False
    return True


def delete_token_at_index(tokens, idx):
    return " ".join(tokens[:idx] + tokens[idx+1:])


def greedy_delete_attack_blackbox(text, max_delete_steps=5, candidate_cap=15):
    """
    Black-box deletion attack:
    At each step, test candidate single-token deletions and keep the one that
    minimizes phishing probability.
    """
    current_text = str(text)

    for _ in range(max_delete_steps):
        tokens = basic_tokenize_with_indices(current_text)

        candidate_indices = [i for i, tok in enumerate(tokens) if is_deletable_token(tok)]

        if not candidate_indices:
            break

        candidate_indices = candidate_indices[:candidate_cap]
        candidate_texts = [delete_token_at_index(tokens, i) for i in candidate_indices]

        _, candidate_probs = batch_predict(candidate_texts)
        best_idx = int(np.argmin(candidate_probs))
        best_text = candidate_texts[best_idx]

        if best_text == current_text:
            break

        current_text = best_text

    return current_text


In [ ]:
def greedy_add_attack_blackbox(text, add_steps=3):
    """
    Greedily applies the benign addition that lowers phishing probability the most.
    """
    current_text = str(text)

    addition_fns = [
        ("benign_prefix", benign_prefix_attack),
        ("benign_suffix", benign_suffix_attack),
        ("contradiction", contradiction_attack),
        ("training_context", training_context_attack),
        ("noise_injection", noise_injection_attack),
    ]

    history = []

    for step in range(1, add_steps + 1):
        candidate_names = []
        candidate_texts = []

        for attack_name, attack_fn in addition_fns:
            candidate_names.append(attack_name)
            candidate_texts.append(attack_fn(current_text))

        candidate_preds, candidate_probs = batch_predict(candidate_texts)

        best_idx = int(np.argmin(candidate_probs))
        current_text = candidate_texts[best_idx]

        history.append({
            "step": step,
            "attack_name": candidate_names[best_idx],
            "pred": int(candidate_preds[best_idx]),
            "phishing_prob": float(candidate_probs[best_idx]),
        })

    return current_text, history


def greedy_delete_subject_body_blackbox(row, max_delete_steps=3, candidate_cap=5):
    sender = row.get("sender", "")
    subject = safe_str(row.get("subject", ""))
    body = safe_str(row.get("body", ""))
    url = safe_str(row.get("url", ""))

    current_subject = subject
    current_body = body

    for _ in range(max_delete_steps):
        subject_tokens = basic_tokenize_with_indices(current_subject)
        body_tokens = basic_tokenize_with_indices(current_body)

        subject_indices = [
            i for i, tok in enumerate(subject_tokens)
            if is_deletable_token(tok)
        ][:candidate_cap]

        remaining_cap = candidate_cap - len(subject_indices)

        body_indices = [
            i for i, tok in enumerate(body_tokens)
            if is_deletable_token(tok)
        ][:max(0, remaining_cap)]

        candidate_texts = []
        candidate_states = []

        for i in subject_indices:
            new_subject = delete_token_at_index(subject_tokens, i)
            candidate_texts.append(
                build_text_all_fields_from_parts(sender, new_subject, current_body, url)
            )
            candidate_states.append((new_subject, current_body))

        for i in body_indices:
            new_body = delete_token_at_index(body_tokens, i)
            candidate_texts.append(
                build_text_all_fields_from_parts(sender, current_subject, new_body, url)
            )
            candidate_states.append((current_subject, new_body))

        if not candidate_texts:
            break

        _, candidate_probs = batch_predict(candidate_texts)
        best_idx = int(np.argmin(candidate_probs))

        best_subject, best_body = candidate_states[best_idx]

        if best_subject == current_subject and best_body == current_body:
            break

        current_subject = best_subject
        current_body = best_body

    return build_text_all_fields_from_parts(sender, current_subject, current_body, url)

def greedy_add_to_body_blackbox(row, add_steps=3):
    sender = row.get("sender", "")
    subject = safe_str(row.get("subject", ""))
    body = safe_str(row.get("body", ""))
    url = safe_str(row.get("url", ""))

    attacked_body, history = greedy_add_attack_blackbox(body, add_steps=add_steps)
    attacked_text = build_text_all_fields_from_parts(sender, subject, attacked_body, url)
    return attacked_text, history


In [ ]:
def add_only_attack(row, add_steps=3):
    attacked_text, _ = greedy_add_to_body_blackbox(row, add_steps=add_steps)
    return attacked_text


def delete_only_attack(row, delete_steps=5):
    attacked_text = greedy_delete_subject_body_blackbox(row, max_delete_steps=delete_steps)
    return attacked_text


def hybrid_add_then_delete_attack_blackbox(row, add_steps=3, delete_steps=5):
    """
    First greedy additions to body, then greedy deletions on subject/body.
    """
    sender = row.get("sender", "")
    subject = safe_str(row.get("subject", ""))
    body = safe_str(row.get("body", ""))
    url = safe_str(row.get("url", ""))

    current_body, _ = greedy_add_attack_blackbox(body, add_steps=add_steps)
    temp_row = {
        "sender": sender,
        "subject": subject,
        "body": current_body,
        "url": url,
    }
    current_text = greedy_delete_subject_body_blackbox(temp_row, max_delete_steps=delete_steps)
    return current_text


In [ ]:
def evaluate_attack_detailed(df_eval, attack_name, row_attack_fn, show_progress=True):
    y_true = df_eval["label_id"].astype(int).tolist()
    texts = df_eval["text"].astype(str).tolist()

    orig_pred, orig_prob = batch_predict(texts)

    attacked_texts = []
    if show_progress:
        iterator = tqdm(df_eval.iterrows(), total=len(df_eval), desc=f"{ACTIVE_MODEL_NAME} | {attack_name}")
    else:
        iterator = df_eval.iterrows()

    for _, row in iterator:
        attacked_texts.append(row_attack_fn(row))

    new_pred, new_prob = batch_predict(attacked_texts)

    metrics = compute_binary_metrics(y_true, new_pred, new_prob)

    details_df = pd.DataFrame({
        "text": texts,
        "label_id": y_true,
        "orig_pred": orig_pred,
        "orig_prob": orig_prob,
        "attacked_text": attacked_texts,
        "new_pred": new_pred,
        "new_prob": new_prob,
    })

    details_df["flipped"] = details_df["orig_pred"] != details_df["new_pred"]
    details_df["prob_drop"] = details_df["orig_prob"] - details_df["new_prob"]

    summary = {
        "attack": attack_name,
        "n_samples": len(df_eval),
        "flip_rate": float(details_df["flipped"].mean()),
        "avg_prob_drop": float(details_df["prob_drop"].mean()),
        **metrics,
    }

    return summary, details_df


In [ ]:
def evaluate_evasion_on_phishing(df_eval, attack_name, row_attack_fn, show_progress=True):
    """
    Restrict evaluation to:
    - true phishing samples
    - originally correct phishing predictions

    Reports attack success rate (ASR).
    """
    df_local = df_eval.copy()
    df_local = df_local[df_local["label_id"].astype(int) == 1].copy()

    if len(df_local) == 0:
        return {
            "attack": attack_name,
            "n_true_phishing": 0,
            "n_orig_correct_phishing": 0,
            "n_flipped": 0,
            "attack_success_rate": float("nan"),
            "robust_recall_on_orig_correct_phishing": float("nan"),
            "avg_prob_drop": float("nan"),
        }, pd.DataFrame()

    texts = df_local["text"].astype(str).tolist()
    orig_pred, orig_prob = batch_predict(texts)

    df_local["orig_pred"] = orig_pred
    df_local["orig_prob"] = orig_prob

    df_attack = df_local[df_local["orig_pred"] == 1].copy()

    if len(df_attack) == 0:
        return {
            "attack": attack_name,
            "n_true_phishing": len(df_local),
            "n_orig_correct_phishing": 0,
            "n_flipped": 0,
            "attack_success_rate": float("nan"),
            "robust_recall_on_orig_correct_phishing": float("nan"),
            "avg_prob_drop": float("nan"),
        }, pd.DataFrame()

    attacked_texts = []
    if show_progress:
        iterator = tqdm(df_attack.iterrows(), total=len(df_attack), desc=f"{ACTIVE_MODEL_NAME} | {attack_name} phishing")
    else:
        iterator = df_attack.iterrows()

    for _, row in iterator:
        attacked_texts.append(row_attack_fn(row))

    new_pred, new_prob = batch_predict(attacked_texts)

    details_df = pd.DataFrame({
        "text": df_attack["text"].astype(str).tolist(),
        "label_id": df_attack["label_id"].astype(int).tolist(),
        "orig_pred": df_attack["orig_pred"].tolist(),
        "orig_prob": df_attack["orig_prob"].tolist(),
        "attacked_text": attacked_texts,
        "new_pred": new_pred,
        "new_prob": new_prob,
    })

    details_df["flipped"] = details_df["new_pred"] != 1
    details_df["prob_drop"] = details_df["orig_prob"] - details_df["new_prob"]

    n_orig_correct = len(details_df)
    n_flipped = int(details_df["flipped"].sum())
    asr = n_flipped / n_orig_correct
    robust_recall = 1.0 - asr

    summary = {
        "attack": attack_name,
        "n_true_phishing": len(df_local),
        "n_orig_correct_phishing": n_orig_correct,
        "n_flipped": n_flipped,
        "attack_success_rate": asr,
        "robust_recall_on_orig_correct_phishing": robust_recall,
        "avg_prob_drop": float(details_df["prob_drop"].mean()),
    }

    return summary, details_df


In [ ]:
val_attack_summaries = []
val_attack_details = {}

attack_specs = [
    ("add_only_add3", lambda row: add_only_attack(row, add_steps=3)),
    ("delete_only_del5", lambda row: delete_only_attack(row, delete_steps=5)),
    ("hybrid_add3_delete5", lambda row: hybrid_add_then_delete_attack_blackbox(row, add_steps=3, delete_steps=5)),
]


In [ ]:
for attack_name, attack_fn in attack_specs:
    summary, details = evaluate_attack_detailed(
        val_df,
        attack_name,
        attack_fn,
        show_progress=True
    )
    val_attack_summaries.append(summary)
    val_attack_details[attack_name] = details

val_attack_results_df = pd.DataFrame(val_attack_summaries).sort_values("f1", ascending=False)
val_attack_results_df

distilbert | add_only_add3:   0%|          | 0/3960 [00:00<?, ?it/s]

distilbert | delete_only_del5:   0%|          | 0/3960 [00:00<?, ?it/s]

distilbert | hybrid_add3_delete5:   0%|          | 0/3960 [00:00<?, ?it/s]

,attack,n_samples,flip_rate,avg_prob_drop,accuracy,precision,recall,f1,roc_auc
1,delete_only_del5,3960,0.008333,0.009402,0.979798,0.984115,0.985606,0.984860,0.998016
0,add_only_add3,3960,0.015909,0.015801,0.975758,0.986239,0.977273,0.981735,0.997753
2,hybrid_add3_delete5,3960,0.034848,0.036027,0.963889,0.992117,0.953409,0.972378,0.996998


In [ ]:
val_evasion_summaries = []
val_evasion_details = {}

for attack_name, attack_fn in attack_specs:
    summary, details = evaluate_evasion_on_phishing(
        val_df,
        attack_name,
        attack_fn,
        show_progress=True
    )
    val_evasion_summaries.append(summary)
    val_evasion_details[attack_name] = details

val_evasion_results_df = pd.DataFrame(val_evasion_summaries).sort_values("attack_success_rate", ascending=False)
val_evasion_results_df

distilbert | add_only_add3 phishing:   0%|          | 0/2618 [00:00<?, ?it/s]

distilbert | delete_only_del5 phishing:   0%|          | 0/2618 [00:00<?, ?it/s]

distilbert | hybrid_add3_delete5 phishing:   0%|          | 0/2618 [00:00<?, ?it/s]

,attack,n_true_phishing,n_orig_correct_phishing,n_flipped,attack_success_rate,robust_recall_on_orig_correct_phishing,avg_prob_drop
2,hybrid_add3_delete5,2640,2618,101,0.038579,0.961421,0.039568
0,add_only_add3,2640,2618,38,0.014515,0.985485,0.014290
1,delete_only_del5,2640,2618,17,0.006494,0.993506,0.006961


In [ ]:
all_test_attack_results = {}
all_test_evasion_results = {}

for test_df in test_dfs:
    dataset_name = test_df["dataset"].iloc[0]

    print(f"\n\n==============================")
    print(f"Starting dataset: {dataset_name}")
    print(f"Rows: {len(test_df)}")
    print(f"==============================")

    attack_summaries = []
    evasion_summaries = []

    for attack_name, attack_fn in attack_specs:
        print(f"\nRunning attack: {attack_name}")

        summary_attack, _ = evaluate_attack_detailed(
            test_df,
            attack_name,
            attack_fn,
            show_progress=True
        )
        attack_summaries.append({"dataset": dataset_name, **summary_attack})

        summary_evasion, _ = evaluate_evasion_on_phishing(
            test_df,
            attack_name,
            attack_fn,
            show_progress=True
        )
        evasion_summaries.append({"dataset": dataset_name, **summary_evasion})

        print("\nAttack metrics:")
        display(pd.DataFrame([{"dataset": dataset_name, **summary_attack}]))

        print("\nPhishing evasion metrics:")
        display(pd.DataFrame([{"dataset": dataset_name, **summary_evasion}]))

    dataset_attack_df = pd.DataFrame(attack_summaries)
    dataset_evasion_df = pd.DataFrame(evasion_summaries)

    all_test_attack_results[dataset_name] = dataset_attack_df
    all_test_evasion_results[dataset_name] = dataset_evasion_df

    print(f"\nFinished dataset: {dataset_name}")

    print("\nAll attack metrics for this dataset:")
    display(dataset_attack_df)

    print("\nAll phishing evasion metrics for this dataset:")
    display(dataset_evasion_df)



Starting dataset: CEAS_08_cleaned
Rows: 39154

Running attack: add_only_add3


distilbert | add_only_add3:   0%|          | 0/39154 [00:00<?, ?it/s]

distilbert | add_only_add3 phishing:   0%|          | 0/13227 [00:00<?, ?it/s]


Attack metrics:


,dataset,attack,n_samples,flip_rate,avg_prob_drop,accuracy,precision,recall,f1,roc_auc
0,CEAS_08_cleaned,add_only_add3,39154,0.212009,0.207641,0.568805,0.966159,0.235281,0.37841,0.951134


...

,dataset,attack,n_true_phishing,n_orig_correct_phishing,n_flipped,attack_success_rate,robust_recall_on_orig_correct_phishing,avg_prob_drop
0,SpamAssasin_cleaned,hybrid_add3_delete5,1718,1032,581,0.562984,0.437016,0.511919



Finished dataset: SpamAssasin_cleaned

All attack metrics for this dataset:


,dataset,attack,n_samples,flip_rate,avg_prob_drop,accuracy,precision,recall,f1,roc_auc
0,SpamAssasin_cleaned,add_only_add3,5809,0.077122,0.076570,0.821656,0.978933,0.405704,0.573663,0.945138
1,SpamAssasin_cleaned,delete_only_del5,5809,0.040282,0.039664,0.848511,0.954447,0.512224,0.666667,0.937888
2,SpamAssasin_cleaned,hybrid_add3_delete5,5809,0.117748,0.118813,0.781374,0.993392,0.262515,0.415285,0.945546



All phishing evasion metrics for this dataset:


,dataset,attack,n_true_phishing,n_orig_correct_phishing,n_flipped,attack_success_rate,robust_recall_on_orig_correct_phishing,avg_prob_drop
0,SpamAssasin_cleaned,add_only_add3,1718,1032,346,0.335271,0.664729,0.303944
1,SpamAssasin_cleaned,delete_only_del5,1718,1032,161,0.156008,0.843992,0.131073
2,SpamAssasin_cleaned,hybrid_add3_delete5,1718,1032,581,0.562984,0.437016,0.511919


In [ ]:
all_test_attack_results = {}
all_test_evasion_results = {}

for test_df in test_dfs:
    dataset_name = test_df["dataset"].iloc[0]

    if dataset_name == "CEAS_08_cleaned":
        print(f"\nSkipping dataset: {dataset_name}")
        continue

    print(f"\n\n==============================")
    print(f"Starting dataset: {dataset_name}")
    print(f"Rows: {len(test_df)}")
    print(f"==============================")

    attack_summaries = []
    evasion_summaries = []

    for attack_name, attack_fn in attack_specs:
        print(f"\nRunning attack: {attack_name}")

        summary_attack, _ = evaluate_attack_detailed(
            test_df,
            attack_name,
            attack_fn,
            show_progress=True
        )
        attack_summaries.append({"dataset": dataset_name, **summary_attack})

        summary_evasion, _ = evaluate_evasion_on_phishing(
            test_df,
            attack_name,
            attack_fn,
            show_progress=True
        )
        evasion_summaries.append({"dataset": dataset_name, **summary_evasion})

        print("\nAttack metrics:")
        display(pd.DataFrame([{"dataset": dataset_name, **summary_attack}]))

        print("\nPhishing evasion metrics:")
        display(pd.DataFrame([{"dataset": dataset_name, **summary_evasion}]))

    dataset_attack_df = pd.DataFrame(attack_summaries)
    dataset_evasion_df = pd.DataFrame(evasion_summaries)

    all_test_attack_results[dataset_name] = dataset_attack_df
    all_test_evasion_results[dataset_name] = dataset_evasion_df

    print(f"\nFinished dataset: {dataset_name}")

    print("\nAll attack metrics for this dataset:")
    display(dataset_attack_df)

    print("\nAll phishing evasion metrics for this dataset:")
    display(dataset_evasion_df)

In [ ]:
combined_attack_df = pd.concat(all_test_attack_results.values(), ignore_index=True)
combined_evasion_df = pd.concat(all_test_evasion_results.values(), ignore_index=True)

print("Combined attack results:")
print(combined_attack_df)

print("\nCombined evasion results:")
print(combined_evasion_df)

import os
os.makedirs("/content/results", exist_ok=True)

val_attack_results_df.to_csv(f"/content/results/{ACTIVE_MODEL_NAME}_val_add_delete_hybrid_metrics.csv", index=False)
val_evasion_results_df.to_csv(f"/content/results/{ACTIVE_MODEL_NAME}_val_add_delete_hybrid_evasion.csv", index=False)
combined_attack_df.to_csv(f"/content/results/{ACTIVE_MODEL_NAME}_test_add_delete_hybrid_metrics.csv", index=False)
combined_evasion_df.to_csv(f"/content/results/{ACTIVE_MODEL_NAME}_test_add_delete_hybrid_evasion.csv", index=False)

print("Saved.")